# Projeto de Integração — VRA/ANAC 2019
## Notebook 01 — Exploração e diagnóstico dos dados (sem limpeza)

**Objetivo:** receber o ZIP original da ANAC, ler os 12 CSVs, preservar os valores e registrar evidências para o relatório.

Este notebook NÃO remove duplicatas, rejeita linhas ou converte os dados para o modelo final. Essas decisões ficarão para as etapas de tratamento, ETL e ELT.

**Fonte:** VRA/ANAC. **Ano:** 2019. **Contrato:** tratamento padronizado do grupo (2019–2021).

### 1. Preparar o ambiente
Usaremos bibliotecas já disponíveis no Google Colab. `dtype=str` evita perder zeros à esquerda de números de voo; `keep_default_na=False` preserva campos vazios sem convertê-los automaticamente.

In [1]:
import io
import re
import zipfile
import pandas as pd
from google.colab import files

print("Ambiente pronto. Pandas:", pd.__version__)

Ambiente pronto. Pandas: 2.2.3


### 2. Carregar o ZIP
Execute e selecione **Base de dados 2019 - ANAC.zip**. Caso envie mais de um ZIP, o código exigirá exatamente um arquivo neste upload para evitar selecionar o incorreto.

In [2]:
arquivos_enviados = files.upload()
zips = [(nome, conteudo) for nome, conteudo in arquivos_enviados.items() if nome.lower().endswith(".zip")]
assert len(zips) == 1, "Envie exatamente um arquivo ZIP nesta execução."
nome_zip, conteudo_zip = zips[0]
assert zipfile.is_zipfile(io.BytesIO(conteudo_zip)), "O arquivo enviado não é um ZIP válido."
print("ZIP carregado:", nome_zip)

Saving Base de dados 2019.zip to Base de dados 2019.zip
ZIP carregado: Base de dados 2019.zip


### 3. Identificar e ler os arquivos sem alterar dados
Adicionamos apenas metadados para auditoria: `arquivo_origem` e `linha_origem`. O número de linha considera cabeçalho na linha 1. **Atenção:** se algum CSV contiver quebras de linha dentro de campos citados, `linha_origem` será a posição do registro lógico, não necessariamente a linha física exata.

In [3]:
partes = []
resumo_mensal = []
cabecalho_referencia = None

with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as z:
    arquivos_csv = sorted(
        nome for nome in z.namelist()
        if re.search(r"VRA_2019_(0[1-9]|1[0-2])\.csv$", nome, re.IGNORECASE)
    )
    assert len(arquivos_csv) == 12, f"Esperados 12 CSVs de 2019; encontrados {len(arquivos_csv)}"
    for nome in arquivos_csv:
        with z.open(nome) as arquivo:
            mensal = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
        assert len(mensal.columns) == 20, f"{nome}: {len(mensal.columns)} colunas, esperado 20"
        if cabecalho_referencia is None:
            cabecalho_referencia = mensal.columns.tolist()
        else:
            assert mensal.columns.tolist() == cabecalho_referencia, f"Cabeçalho divergente: {nome}"
        mensal["arquivo_origem"] = nome
        mensal["linha_origem"] = range(2, len(mensal) + 2)
        resumo_mensal.append({"arquivo": nome, "registros": len(mensal)})
        partes.append(mensal)

df_2019 = pd.concat(partes, ignore_index=True)
colunas_originais = cabecalho_referencia
print("Arquivos:", len(arquivos_csv))
print("Registros:", len(df_2019))
print("Colunas da fonte:", len(colunas_originais))
print("Colunas com rastreabilidade:", len(df_2019.columns))
display(pd.DataFrame(resumo_mensal))
display(df_2019.head(3))

Arquivos: 12
Registros: 983414
Colunas da fonte: 20
Colunas com rastreabilidade: 22


,arquivo,registros
0,Base de dados 2019/VRA_2019_01.csv,91914
1,Base de dados 2019/VRA_2019_02.csv,79571
2,Base de dados 2019/VRA_2019_03.csv,83152
3,Base de dados 2019/VRA_2019_04.csv,77445
4,Base de dados 2019/VRA_2019_05.csv,76732
5,Base de dados 2019/VRA_2019_06.csv,74254
6,Base de dados 2019/VRA_2019_07.csv,85687
7,Base de dados 2019/VRA_2019_08.csv,80588
8,Base de dados 2019/VRA_2019_09.csv,80537
9,Base de dados 2019/VRA_2019_10.csv,84268


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,...,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem
0,AAF,AIGLE AZUR,0035,0,I,A332,283,LFPO,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,25/01/2019 06:15,...,VIRACOPOS - CAMPINAS - SP - BRASIL,25/01/2019 18:15,25/01/2019 18:15,REALIZADO,,2019-01-25,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,2
1,AAF,AIGLE AZUR,0036,0,I,A332,283,SBKP,VIRACOPOS - CAMPINAS - SP - BRASIL,25/01/2019 20:15,...,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,26/01/2019 07:45,26/01/2019 07:45,REALIZADO,,2019-01-25,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,3
2,AAF,AIGLE AZUR,0037,0,I,A332,283,LFPO,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,26/01/2019 09:45,...,VIRACOPOS - CAMPINAS - SP - BRASIL,26/01/2019 21:45,26/01/2019 21:45,REALIZADO,,2019-01-26,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,4


### 4. Conferir campos vazios
Contamos strings vazias ou compostas apenas de espaços, **sem alterar o DataFrame**.

In [5]:
# Conta os valores vazios de cada coluna original
vazios = df_2019[colunas_originais].apply(
    lambda c: c.str.strip().eq("").sum()
)

# Organiza os resultados em uma tabela
resumo_vazios = vazios.reset_index()

# Renomeia as colunas
resumo_vazios.columns = ["coluna", "registros_vazios"]

# Ordena da coluna com mais valores vazios para a com menos
resumo_vazios = resumo_vazios.sort_values(
    "registros_vazios",
    ascending=False
)

display(resumo_vazios)

,coluna,registros_vazios
16,Justificativa,642482
18,Situação Partida,18153
10,Partida Real,18153
14,Chegada Real,18153
19,Situação Chegada,18153
3,Código DI,612
13,Chegada Prevista,18
9,Partida Prevista,18
11,Sigla ICAO Aeroporto Destino,14
6,Número de Assentos,11


### 5. Diagnosticar duplicatas exatas
O critério é comparar as **20 colunas originais** em toda a base anual. Ignoramos `arquivo_origem` e `linha_origem`, pois cada registro tem origem própria. Nenhuma linha será eliminada aqui.

In [6]:
duplicatas_excedentes = int(df_2019.duplicated(subset=colunas_originais, keep="first").sum())
registros_envolvidos = int(df_2019.duplicated(subset=colunas_originais, keep=False).sum())
grupos_repetidos = registros_envolvidos - duplicatas_excedentes
print("Grupos de registros repetidos:", grupos_repetidos)
print("Registros envolvidos em duplicatas:", registros_envolvidos)
print("Duplicatas excedentes:", duplicatas_excedentes)
print("Total hipotético após deduplicação:", len(df_2019) - duplicatas_excedentes)

Grupos de registros repetidos: 444
Registros envolvidos em duplicatas: 914
Duplicatas excedentes: 470
Total hipotético após deduplicação: 982944


### 6. Investigar registros sem datas previstas
O contrato exige partida e chegada **previstas** para carga na fato. Primeiro contamos ausências de qualquer uma delas; também verificamos separadamente linhas sem nenhuma das quatro datas de partida/chegada.

In [7]:
def ausente(coluna):
    return df_2019[coluna].str.strip().eq("")

sem_previstas = ausente("Partida Prevista") | ausente("Chegada Prevista")
sem_quatro_datas = (ausente("Partida Prevista") & ausente("Partida Real")
                   & ausente("Chegada Prevista") & ausente("Chegada Real"))
print("Sem alguma data prevista obrigatória:", int(sem_previstas.sum()))
print("Sem nenhuma das quatro datas:", int(sem_quatro_datas.sum()))
print("Situações dos registros com datas previstas ausentes:")
display(df_2019.loc[sem_previstas, "Situação Voo"].value_counts(dropna=False).rename_axis("situacao").reset_index(name="registros"))

# Interseção usada para evitar subtrair duas vezes o mesmo registro
excedente = df_2019.duplicated(subset=colunas_originais, keep="first")
rejeitadas_apos_deduplicacao = int((sem_previstas & ~excedente).sum())
print("Rejeições após deduplicar (sem sobreposição):", rejeitadas_apos_deduplicacao)
print("Contagem analítica estimada:", len(df_2019)-duplicatas_excedentes-rejeitadas_apos_deduplicacao)

Sem alguma data prevista obrigatória: 18
Sem nenhuma das quatro datas: 18
Situações dos registros com datas previstas ausentes:


,situacao,registros
0,REALIZADO,14
1,CANCELADO,2
2,NÃO INFORMADO,2


Rejeições após deduplicar (sem sobreposição): 18
Contagem analítica estimada: 982926


### 7. Resumo para o relatório
Registre os números exibidos aqui e as decisões no relatório. **Ainda não transformamos nem carregamos o banco.**

In [8]:
diagnostico = pd.DataFrame([
    ("ano", 2019),
    ("arquivos", len(arquivos_csv)),
    ("registros_brutos", len(df_2019)),
    ("colunas_originais", len(colunas_originais)),
    ("duplicatas_envolvidos", registros_envolvidos),
    ("duplicatas_excedentes", duplicatas_excedentes),
    ("sem_alguma_data_prevista", int(sem_previstas.sum())),
    ("sem_quatro_datas", int(sem_quatro_datas.sum())),
    ("rejeitadas_apos_deduplicacao", rejeitadas_apos_deduplicacao),
    ("registros_analiticos_estimados", len(df_2019)-duplicatas_excedentes-rejeitadas_apos_deduplicacao),
], columns=["indicador", "valor"])
display(diagnostico)
print("Etapa de exploração concluída. Os dados originais permanecem sem limpeza.")

,indicador,valor
0,ano,2019
1,arquivos,12
2,registros_brutos,983414
3,colunas_originais,20
4,duplicatas_envolvidos,914
5,duplicatas_excedentes,470
6,sem_alguma_data_prevista,18
7,sem_quatro_datas,18
8,rejeitadas_apos_deduplicacao,18
9,registros_analiticos_estimados,982926


Etapa de exploração concluída. Os dados originais permanecem sem limpeza.


## Próxima etapa
Após revisar este diagnóstico, implementaremos **o pipeline responsável (ETL com Pandas ou ELT com SQL)** conforme a divisão do grupo. As remoções serão feitas somente na camada tratada, mantendo a base bruta intacta.